# Django Rest Framework

Ejercicio de implementación de ViewSets para una API de eCommerce.

Se implementaron los métodos: create, list, retrieve, update, partial_update y destroy.

## Serializer

In [ ]:
from rest_framework import serializers

from .models import ProductModel


class ProductSerializer(serializers.ModelSerializer):

    class Meta:
        model = ProductModel
        fields = [
            "id",
            "name",
            "price",
            "description",
            "seller",
            "color",
            "product_dimensions",
        ]
        read_only_fields = ["id"]

## ProductViewSet

In [ ]:
from rest_framework import status, viewsets
from rest_framework.response import Response

from .models import ProductModel
from .serializers import ProductSerializer


class ProductViewSet(viewsets.ModelViewSet):
    queryset = ProductModel.objects.all().order_by("id")
    serializer_class = ProductSerializer

    def create(self, request, *args, **kwargs):
        print("ViewSet CREATE ejecutado")

        serializer = self.get_serializer(data=request.data)
        serializer.is_valid(raise_exception=True)
        self.perform_create(serializer)

        return Response(
            serializer.data,
            status=status.HTTP_201_CREATED,
        )

    def list(self, request, *args, **kwargs):
        print("ViewSet LIST ejecutado")

        queryset = self.filter_queryset(self.get_queryset())
        serializer = self.get_serializer(queryset, many=True)

        return Response(serializer.data)

    def retrieve(self, request, *args, **kwargs):
        print("ViewSet RETRIEVE ejecutado")

        instance = self.get_object()
        serializer = self.get_serializer(instance)

        return Response(serializer.data)

    def update(self, request, *args, **kwargs):
        print("ViewSet UPDATE ejecutado")

        partial = kwargs.pop("partial", False)
        instance = self.get_object()

        serializer = self.get_serializer(
            instance,
            data=request.data,
            partial=partial,
        )

        serializer.is_valid(raise_exception=True)
        self.perform_update(serializer)

        return Response(serializer.data)

    def partial_update(self, request, *args, **kwargs):
        print("ViewSet PARTIAL_UPDATE ejecutado")

        kwargs["partial"] = True

        return self.update(
            request,
            *args,
            **kwargs,
        )

    def destroy(self, request, *args, **kwargs):
        print("ViewSet DESTROY ejecutado")

        instance = self.get_object()
        self.perform_destroy(instance)

        return Response(
            {"message": "Producto eliminado correctamente"},
            status=status.HTTP_200_OK,
        )

## Configuración

Se agregó `rest_framework` a `INSTALLED_APPS`.

In [ ]:
INSTALLED_APPS = [
    "rest_framework",
    "ejercicios",
]

## Router

In [ ]:
from django.urls import include, path
from rest_framework.routers import DefaultRouter

from ejercicios.viewsets import ProductViewSet


router = DefaultRouter()

router.register(
    "products",
    ProductViewSet,
    basename="viewset-products",
)

urlpatterns = [
    path("api/viewset/", include(router.urls)),
]

# Pruebas realizadas

## LIST
Se consultó la lista de productos y se obtuvieron correctamente
los registros almacenados en la base de datos.

## RETRIEVE
Se consultó el producto con ID 2.

Resultado:
ID: 2
Nombre: Producto 1
Precio: 101.00

## CREATE
Se creó un producto mediante POST.

Resultado:
ID: 503
Nombre: Producto ViewSet
Precio: 899.99
Color: Rojo

## UPDATE
Se actualizó completamente el producto 503 mediante PUT.

Resultado:
Nombre: Producto ViewSet Actualizado
Precio: 999.99
Color: Azul

## PARTIAL_UPDATE
Se modificaron únicamente el precio y el color mediante PATCH.

Resultado:
Precio: 1099.99
Color: Negro

Los demás atributos conservaron sus valores anteriores.

## DESTROY
Se eliminó el producto 503 mediante DELETE.

Resultado:
Producto eliminado correctamente.

Finalmente se intentó consultar nuevamente el producto 503
y el servidor respondió HTTP 404 Not Found.

Esto confirmó que el producto fue eliminado de la base de datos.


# Conclusión

Se implementaron y probaron correctamente los métodos
create, list, retrieve, update, partial_update y destroy
utilizando Django REST Framework y un ModelViewSet.
